# Amphion — ESM-2 upgrade (Phase 2B / 3-ESM)

Replaces the 29 hand-crafted features with **ESM-2 protein-language-model embeddings**
and retrains the activity, MIC, and toxicity models — on the **exact same** sequences,
labels, and homology clusters as the baseline, so the comparison is honest.

### ▶ How to run (Kaggle)
1. **Upload the inputs as a Dataset:** create a Kaggle Dataset from the repo's
   `data/external/esm_inputs/` folder (the 3 CSVs: `activity.csv`, `mic.csv`, `hemolysis.csv`),
   then **Add Input → your dataset** to this notebook.
2. **Settings:** Accelerator = **GPU**, Internet = **ON** (needed to download ESM-2 weights).
3. **Run All** (~3–5 min).
4. **Download** from `/kaggle/working/`: `activity_clf_esm.joblib`, `mic_reg_esm.joblib`,
   `tox_clf_esm.joblib`, `esm_config.json`, `metrics_esm.json`.
5. Put the joblibs + `esm_config.json` in the repo's `models/` and tell Claude Code to integrate.

> Predictions with uncertainty, not proof.


In [ ]:
import os, sys, json, glob, time, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fair-esm'], check=True)

import numpy as np, pandas as pd, torch
np.random.seed(42); torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', device)
if device == 'cpu':
    print('WARNING: no GPU — enable the GPU accelerator for a fast run.')

# ESM-2 model: t30_150M is a strong accuracy/size balance and runs on a free CPU at inference.
# Lighter alternative for a snappier demo: esm2_t12_35M_UR50D (layer 12, dim 480).
ESM_MODEL = 'esm2_t30_150M_UR50D'
REPR_LAYER = 30
BASELINE = {'activity_AUC': 0.954, 'activity_MCC': 0.781,
            'mic_RMSE': 0.755, 'mic_Spearman': 0.518,
            'tox_ROC_AUC': 0.762, 'tox_PR_AUC': 0.704}

## 1. Load the uploaded datasets (identical to the repo's processed data)

In [ ]:
def find_csv(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits:
        raise FileNotFoundError(f'{name} not found under /kaggle/input — did you Add Input -> your dataset?')
    return hits[0]

act = pd.read_csv(find_csv('activity.csv'))
mic = pd.read_csv(find_csv('mic.csv'))
hemo = pd.read_csv(find_csv('hemolysis.csv'))
print('activity', act.shape, '| mic', mic.shape, '| hemolysis', hemo.shape)

all_seqs = sorted(set(act.sequence) | set(mic.sequence) | set(hemo.sequence))
print('unique sequences to embed:', len(all_seqs))

## 2. Load ESM-2 and embed every sequence (mean-pooled residue embeddings)

In [ ]:
import esm
model, alphabet = getattr(esm.pretrained, ESM_MODEL)()
model = model.eval().to(device)
bc = alphabet.get_batch_converter()
EMBED_DIM = model.embed_dim if hasattr(model, 'embed_dim') else model.args.embed_dim
print('ESM model:', ESM_MODEL, '| layers:', REPR_LAYER, '| embed dim:', EMBED_DIM)

@torch.no_grad()
def embed_batch(seqs):
    data = [(str(i), s) for i, s in enumerate(seqs)]
    _, _, toks = bc(data)
    toks = toks.to(device)
    rep = model(toks, repr_layers=[REPR_LAYER])['representations'][REPR_LAYER]
    vecs = []
    for k, s in enumerate(seqs):
        L = len(s)
        vecs.append(rep[k, 1:L + 1].mean(0).float().cpu().numpy())  # exclude BOS/EOS/pad
    return np.asarray(vecs, dtype=np.float32)

t0 = time.time(); BATCH = 32
emb = {}
for i in range(0, len(all_seqs), BATCH):
    chunk = all_seqs[i:i + BATCH]
    for s, v in zip(chunk, embed_batch(chunk)):
        emb[s] = v
    if (i // BATCH) % 25 == 0:
        print(f'  embedded {min(i+BATCH, len(all_seqs))}/{len(all_seqs)}')
print(f'embedded {len(emb)} sequences in {time.time()-t0:.0f}s')

def X_of(seqs):
    return np.vstack([emb[s] for s in seqs])

## 3. Train + evaluate (cluster-aware) on ESM embeddings — vs the 29-feature baseline

In [ ]:
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, matthews_corrcoef, average_precision_score,
                             mean_squared_error, accuracy_score, f1_score)
from scipy.stats import spearmanr

def clf_models():
    return {'LogReg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=1.0)),
            'MLP': make_pipeline(StandardScaler(), MLPClassifier((256,), max_iter=400, random_state=42))}

def clf_models_balanced():
    return {'LogReg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced')),
            'MLP': make_pipeline(StandardScaler(), MLPClassifier((256,), max_iter=400, random_state=42))}

# --- Activity (balanced subset) ---
a = act[act.in_balanced].reset_index(drop=True)
Xa, ya, ga = X_of(a.sequence.tolist()), a.label.to_numpy(), a.cluster_id.to_numpy()
sgkf = StratifiedGroupKFold(5)
best_a, best_a_auc = None, -1
for name, m in clf_models().items():
    p = cross_val_predict(m, Xa, ya, cv=sgkf, groups=ga, method='predict_proba')[:, 1]
    auc, mcc = roc_auc_score(ya, p), matthews_corrcoef(ya, (p >= .5).astype(int))
    print(f'[activity] {name:7s} AUC={auc:.3f} MCC={mcc:.3f}')
    if auc > best_a_auc: best_a, best_a_auc, best_a_name = m, auc, name
act_metrics = {'AUC': float(best_a_auc), 'model': best_a_name}

# --- MIC regression ---
Xm, ym, gm = X_of(mic.sequence.tolist()), mic.min_log_mic_uM.to_numpy(), mic.cluster_id.to_numpy()
gkf = GroupKFold(5)
best_m, best_m_rmse = None, 1e9
for name, m in {'Ridge': make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
                'MLP': make_pipeline(StandardScaler(), MLPRegressor((256,), max_iter=400, random_state=42))}.items():
    p = cross_val_predict(m, Xm, ym, cv=gkf, groups=gm)
    rmse, sp = float(np.sqrt(mean_squared_error(ym, p))), float(spearmanr(ym, p).statistic)
    print(f'[mic]      {name:7s} RMSE={rmse:.3f} Spearman={sp:.3f}')
    if rmse < best_m_rmse: best_m, best_m_rmse, best_m_name, best_m_sp = m, rmse, name, sp
mic_metrics = {'RMSE': best_m_rmse, 'Spearman': best_m_sp, 'model': best_m_name}

# --- Toxicity ---
Xt, yt, gt = X_of(hemo.sequence.tolist()), hemo.hemolytic.to_numpy(), hemo.cluster_id.to_numpy()
best_t, best_t_pr = None, -1
for name, m in clf_models_balanced().items():
    p = cross_val_predict(m, Xt, yt, cv=sgkf, groups=gt, method='predict_proba')[:, 1]
    roc, pr = roc_auc_score(yt, p), average_precision_score(yt, p)
    print(f'[toxicity] {name:7s} ROC-AUC={roc:.3f} PR-AUC={pr:.3f}')
    if pr > best_t_pr: best_t, best_t_pr, best_t_name, best_t_roc = m, pr, name, roc
tox_metrics = {'ROC_AUC': best_t_roc, 'PR_AUC': best_t_pr, 'model': best_t_name}

print('\n=== ESM-2 vs baseline (cluster-aware) ===')
print(f"activity AUC : {act_metrics['AUC']:.3f}  (baseline {BASELINE['activity_AUC']:.3f})")
print(f"mic RMSE     : {mic_metrics['RMSE']:.3f}  (baseline {BASELINE['mic_RMSE']:.3f}; lower better)")
print(f"mic Spearman : {mic_metrics['Spearman']:.3f}  (baseline {BASELINE['mic_Spearman']:.3f})")
print(f"tox ROC-AUC  : {tox_metrics['ROC_AUC']:.3f}  (baseline {BASELINE['tox_ROC_AUC']:.3f})")
print(f"tox PR-AUC   : {tox_metrics['PR_AUC']:.3f}  (baseline {BASELINE['tox_PR_AUC']:.3f})")

## 4. Fit final models on all data + save (calibrated classifiers)

In [ ]:
import joblib
from sklearn.calibration import CalibratedClassifierCV

OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
esm_cfg = {'model_id': ESM_MODEL, 'repr_layer': REPR_LAYER, 'embed_dim': int(EMBED_DIM), 'pooling': 'mean'}

act_final = CalibratedClassifierCV(best_a, method='isotonic', cv=5).fit(Xa, ya)
joblib.dump({'kind': 'activity_clf', 'features': 'esm', 'esm': esm_cfg, 'model': act_final,
             'metrics': {'cluster_aware': {'AUC': act_metrics['AUC']}}}, f'{OUT}/activity_clf_esm.joblib')

mic_final = best_m.fit(Xm, ym)
joblib.dump({'kind': 'mic_reg', 'features': 'esm', 'esm': esm_cfg, 'model': mic_final,
             'metrics': mic_metrics}, f'{OUT}/mic_reg_esm.joblib')

tox_final = CalibratedClassifierCV(best_t, method='isotonic', cv=5).fit(Xt, yt)
joblib.dump({'kind': 'tox_clf', 'features': 'esm', 'esm': esm_cfg, 'model': tox_final,
             'metrics': tox_metrics}, f'{OUT}/tox_clf_esm.joblib')

json.dump(esm_cfg, open(f'{OUT}/esm_config.json', 'w'), indent=2)
json.dump({'esm': {'activity': act_metrics, 'mic': mic_metrics, 'toxicity': tox_metrics},
           'baseline': BASELINE}, open(f'{OUT}/metrics_esm.json', 'w'), indent=2)
print('SAVED to', OUT, ':')
for f in ['activity_clf_esm.joblib', 'mic_reg_esm.joblib', 'tox_clf_esm.joblib', 'esm_config.json', 'metrics_esm.json']:
    print('  ', f)
print('\nDownload these into the repo models/ and tell Claude Code to integrate.')